In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/mohs_hardness/train.csv')

# Display the first few rows of the dataset
print(train_data.head())

# Get a summary of the dataset
print(train_data.info())

# Check for missing values
print(train_data.isnull().sum())

# Check for duplicate rows
print(train_data.duplicated().sum())

# Distinguish column types
numeric_cols = train_data.select_dtypes(include=[np.number]).columns
categorical_cols = train_data.select_dtypes(include=['object', 'category']).columns

print("Numeric Columns:", numeric_cols)
print("Categorical Columns:", categorical_cols)

# Visualize the distribution of numeric columns
for col in numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(train_data[col], bins=30, kde=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()

# Visualize the distribution of categorical columns
for col in categorical_cols:
    plt.figure(figsize=(8, 4))
    sns.countplot(y=train_data[col])
    plt.title(f'Distribution of {col}')
    plt.xlabel('Count')
    plt.ylabel(col)
    plt.show()

# Check for correlations among numeric columns
plt.figure(figsize=(12, 8))
correlation_matrix = train_data[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.show()


     id  allelectrons_Total  ...  density_Average  Hardness
0  2124                30.0  ...          0.51006       6.0
1   394                64.0  ...          4.74000       3.3
2  3101                97.0  ...          1.79976       5.3
3  1737               151.0  ...          7.77500       1.8
4   561               131.0  ...          1.92652       5.5

[5 rows x 13 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8325 entries, 0 to 8324
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     8325 non-null   int64  
 1   allelectrons_Total     8325 non-null   float64
 2   density_Total          8325 non-null   float64
 3   allelectrons_Average   8325 non-null   float64
 4   val_e_Average          8325 non-null   float64
 5   atomicweight_Average   8325 non-null   float64
 6   ionenergy_Average      8325 non-null   float64
 7   el_neg_chi_Average     8325 non-null 

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data)
print("column_info")
print(column_info)


2025-09-15 00:22:59.137 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': [], 'Numeric': ['id', 'allelectrons_Total', 'density_Total', 'allelectrons_Average', 'val_e_Average', 'atomicweight_Average', 'ionenergy_Average', 'el_neg_chi_Average', 'R_vdw_element_Average', 'R_cov_element_Average', 'zaratio_Average', 'density_Average', 'Hardness'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, StandardScale

# Load the training data
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/mohs_hardness/train.csv')

# Separate features and target
X_train = train_data.drop(columns=['Hardness'])
y_train = train_data['Hardness']

# Handle missing values
fill_missing = FillMissingValue(features=X_train.columns, strategy='mean')
X_train = fill_missing.fit_transform(X_train)

# Normalize numerical features
standard_scale = StandardScale(features=X_train.columns)
X_train = standard_scale.fit_transform(X_train)

# Load the test data
test_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/mohs_hardness/test.csv')

# Separate features and target
X_test = test_data.drop(columns=['Hardness'])
y_test = test_data['Hardness']

# Handle missing values on test data
X_test = fill_missing.transform(X_test)

# Normalize numerical features on test data
X_test = standard_scale.transform(X_test)


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(X_train)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['id', 'allelectrons_Total', 'density_Total', 'allelectrons_Average', 'val_e_Average', 'atomicweight_Average', 'ionenergy_Average', 'el_neg_chi_Average', 'R_vdw_element_Average', 'R_cov_element_Average', 'zaratio_Average', 'density_Average'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

# Assuming X_train, y_train, X_test, y_test are already defined from previous tasks

# Check for any categorical columns (though we expect none based on the latest data info)
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns

# If there are categorical columns, encode them
if not categorical_cols.empty:
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        X_train[col] = le.fit_transform(X_train[col])
        X_test[col] = le.transform(X_test[col])
        label_encoders[col] = le

# Initialize and train the model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Calculate the Median Absolute Error (MAE)
mae = mean_absolute_error(y_test, y_pred)
print(f"Median Absolute Error: {mae}")


Median Absolute Error: 0.9198340537944283


In [6]:
import pandas as pd
from sklearn.metrics import median_absolute_error

# Load the test data
test_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/mohs_hardness/test.csv')

# Separate features and target
X_test = test_data.drop(columns=['Hardness'])
y_test = test_data['Hardness']

# Transform the test data using the same preprocessing steps as the training data
X_test = fill_missing.transform(X_test)
X_test = standard_scale.transform(X_test)

# Ensure categorical columns are encoded using the same label encoders
categorical_cols = X_test.select_dtypes(include=['object', 'category']).columns
if not categorical_cols.empty:
    for col in categorical_cols:
        X_test[col] = label_encoders[col].transform(X_test[col])

# Predict using the trained model
y_pred = model.predict(X_test)

# Calculate and print the Median Absolute Error
mae = median_absolute_error(y_test, y_pred)
print(f"Median Absolute Error: {mae}")


Median Absolute Error: 0.6850000000000016
